In [1]:
pip install -qq langchain langchain-openai langchain-community chromadb sentence-transformers openai pypdf tiktoken dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

# API_KEY = "YOUR_OPENAI_API_KEY_HERE"
# os.environ["OPENAI_API_KEY"] = API_KEY

                                                 
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
print(API_KEY) 

qweqweYOUR_OPENAI_API_KEY_HERE


# Datasource

In [28]:
import urllib.request

url = "https://isap.sejm.gov.pl/isap.nsf/download.xsp/WDU19970780483/U/D19970483Lj.pdf"
urllib.request.urlretrieve(url, "konstytucja.pdf")

('konstytucja.pdf', <http.client.HTTPMessage at 0x163f9a520>)

In [ ]:
from pypdf import PdfReader

reader = PdfReader("konstytucja.pdf")
raw_text = "\n".join(page.extract_text() for page in reader.pages)
print(len(raw_text))

108868


: 

# Index pipeline

In [5]:
import re

# Based on document structure, find best split for chunks
chunks = re.split(r'(Art\. \d+\.)', raw_text)

# Add source to chunk
articles = []
for i in range(1, len(chunks), 2):
    header = chunks[i].strip()
    body = chunks[i+1].strip() if i+1 < len(chunks) else ""
    articles.append({"text": f"{header} {body}", "source": header})

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

docs = [Document(page_content=a["text"], metadata={"source": a["source"]}) for a in articles]
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=API_KEY)

vectorstore = Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db2")

/opt/miniconda3/envs/py3135/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Ile dokumentów zapisało się w ChromaDB?
print(f"Dokumentów w ChromaDB: {vectorstore._collection.count()}")

# Czy embeddingi mają sens? Szybki test wyszukiwania
wyniki = vectorstore.similarity_search("Zgromadzenie i sejm", k=5)
for w in wyniki:
    print(w.metadata["source"])
    print(w.page_content[:300])
    print("---")

Dokumentów w ChromaDB: 972
Art. 96.
Art. 96. 1. Sejm składa się z 460 posłów. 
2. Wybory do Sejmu są powszechne, równe, bezpośrednie i proporcjonalne 
oraz odbywają się w głosowaniu tajnym.
---
Art. 96.
Art. 96. 1. Sejm składa się z 460 posłów. 
2. Wybory do Sejmu są powszechne, równe, bezpośrednie i proporcjonalne 
oraz odbywają się w głosowaniu tajnym.
---
Art. 96.
Art. 96. 1. Sejm składa się z 460 posłów. 
2. Wybory do Sejmu są powszechne, równe, bezpośrednie i proporcjonalne 
oraz odbywają się w głosowaniu tajnym.
---
Art. 96.
Art. 96. 1. Sejm składa się z 460 posłów. 
2. Wybory do Sejmu są powszechne, równe, bezpośrednie i proporcjonalne 
oraz odbywają się w głosowaniu tajnym.
---
Art. 114.
Art. 114. 1. W przypadkach określonych w Konstytucji Sejm i Senat, obradując 
wspólnie pod przewodnictwem Marszałka Sejmu lub w jego zastępstwie Marszałka 
Senatu, działają jako Zgromadzenie Narodowe. 
2. Zgromadzenie Narodowe uchwala swój regulamin.
---


# Question pipeline

In [8]:
from langchain.agents import create_agent
from langchain.tools import tool

In [9]:
@tool(response_format="content_and_artifact")
def retrieve_constitution_content(query: str):
    """
    Wyszukuje informacje zawarte w konstytucji RP.

    Args:
        query: Pytanie lub fraza do wyszukania

    Returns:
        Tuple zawierajacy sformatowana odpowiedz i oryginalne dokumenty
    """
    retrieved_docs = vectorstore.similarity_search(query, k=1)
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs



# OpenAI LLM

In [10]:
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-4o-mini"

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, temperature=0.0)

In [11]:
rag_agent = create_agent(
    model=llm,
    tools=[retrieve_constitution_content],
    system_prompt=(
        "Jestes ekspertem od Konstytucji RP."
        "Odpowiadaj WYLACZNIE na podstawie podanego kontekstu."
        "Zawsze podawaj numer artykulu."
        "Jesli nie znajdziesz odpowiedzi — powiedz"
        "'Nie znalazlem odpowiedzi w Konstytucji.'"        
    ),
)

In [12]:
import textwrap

def ask_lawyer(q):
    result = rag_agent.invoke({
        "messages": [{"role": "user", "content": q}]
    })
    answer = result["messages"][-1].content
    print(f"Pytanie: {q}")
    print("Odpowiedź:", textwrap.fill(answer, width=80))
    print("-" * 80)

In [16]:
answer = ask_lawyer("Jak zrobić naleśniki?")





Pytanie: Jak zrobić naleśniki?
Odpowiedź: Nie znalazłem odpowiedzi w Konstytucji.
--------------------------------------------------------------------------------


In [ ]:
ask_lawyer("Kto powołuje premiera?")
ask_lawyer("Ile trwa kadencja Sejmu?")
ask_lawyer("Jaka jest stolica Francji?")

Pytanie: Kto powołuje premiera?
Odpowiedź: Premiera powołuje Prezydent Rzeczypospolitej, który desygnuje Prezesa Rady
Ministrów. Prezydent powołuje Prezesa Rady Ministrów wraz z pozostałymi
członkami Rady Ministrów w ciągu 14 dni od dnia pierwszego posiedzenia Sejmu lub
przyjęcia dymisji poprzedniej Rady Ministrów. (Art. 154)
--------------------------------------------------------------------------------
Pytanie: Ile trwa kadencja Sejmu?
Odpowiedź: Kadencja Sejmu trwa cztery lata. (Art. 98)
--------------------------------------------------------------------------------
Pytanie: Jaka jest stolica Francji?
Odpowiedź: Nie znalazłem odpowiedzi w Konstytucji.
--------------------------------------------------------------------------------


: 

In [ ]:
rag_agent.invoke({
        "messages": [{"role": "user", "content": "Ile trwa kadencja Sejmu?"}]
    })

{'messages': [HumanMessage(content='Ile trwa kadencja Sejmu?', additional_kwargs={}, response_metadata={}, id='2b0eb06a-e90c-44bc-afb5-298d05d2f5cb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 164, 'total_tokens': 186, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0a4fd4d629', 'id': 'chatcmpl-DZKfVGjzAB6rRxgIcrUP5gm6VNTek', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd020-c336-7bc2-9a49-7c35b0b6517d-0', tool_calls=[{'name': 'retrieve_constitution_content', 'args': {'query': 'kadencja Sejmu'}, 'id': 'call_Tiw03fw9GoFr7OvsIwfV7yTB', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

: 